# Does self-attention actually capture context effects? — CDM synthetic test

This notebook builds a synthetic choice dataset from a **Context Dependent Model (CDM)** where the true context effect is known exactly, then tests how much of that effect a flexible **self-attention** model actually recovers.

**The logic in one line:** plant a known pairwise context effect → confirm a correctly-specified model recovers it perfectly (the ceiling) → measure how much the attention model recovers → validate the measurement on a no-effect control → repeat across seeds for error bars.

**Headline result:** the attention model recovers only **~46%** of the true pairwise effect (slope 0.463 ± 0.018) despite ~81% click-prediction accuracy — capacity is not the bottleneck, identification is.

## 1. Imports and softmax helper

We only need `numpy` and `pandas` to build the dataset. The `softmax` turns a set of item utilities into choice probabilities that sum to 1 — the max-subtraction is just for numerical stability (stops `exp` from overflowing). This is the function that converts "how good is each item" into "how likely is each item to be clicked".

In [ ]:
import numpy as np
import pandas as pd

def softmax(u):
    u = u - np.max(u)
    e = np.exp(u)
    return e / e.sum()

## 2. The CDM data generator

This is the heart of the synthetic setup. For every item *i* in a slate *S* the utility is:

$$U(i, S) = \theta^\top x_i \;+\; \sum_{j \in S,\, j \neq i} x_j^\top C\, x_i$$

- The first term $\theta^\top x_i$ is the plain MNL part — how appealing the item is on its own features.
- The second term is the **CDM context effect**: every *other* item *j* in the slate pushes on *i* through the interaction matrix **C**. This is a sum of *pairwise* pushes, which is exactly what a set-average (LCL) cannot reproduce.

`context_strength` is the single knob: set it to 0 and **C** vanishes, giving pure MNL with no context effect (our baseline). The matrix **C** is drawn once from a fixed RNG (seed 12345) so the ground truth is identical every run. We then softmax the utilities and sample one winner per slate.

In [ ]:
# Model, for item i in slate S:
#     U(i, S) = theta . x_i  +  sum_{j != i} x_j^T C x_i
# Knob: context_strength=0 -> C=0 -> pure MNL baseline (no context).
def generate_cdm_data(n_sessions=8000, n_features=3, slate_size=5,
                      theta=None, C=None, context_strength=1.0,
                      feature_scale=1.0, seed=0):
    rng = np.random.default_rng(seed)
    d = n_features

    if theta is None:
        base = np.array([-1.0, 1.0, 0.5])
        theta = base[:d] if d <= 3 else np.concatenate([base, np.zeros(d - 3)])
    theta = np.asarray(theta, dtype=float)

    if C is None:
        c_rng = np.random.default_rng(12345)   # fixed -> reproducible C
        C = c_rng.normal(0.0, 1.0, size=(d, d))
    C = np.asarray(C, dtype=float)
    C_eff = context_strength * C

    rows = []
    for s in range(n_sessions):
        X = rng.normal(0.0, feature_scale, size=(slate_size, d))
        U = X @ theta
        M = X @ C_eff @ X.T                       # M[j,i] = x_j^T C x_i
        context = M.sum(axis=0) - np.diag(M)      # sum over j != i
        U = U + context
        P = softmax(U)
        chosen_idx = rng.choice(slate_size, p=P)
        for i in range(slate_size):
            row = {"session_id": s, "item_in_slate": i}
            for f in range(d):
                row[f"feat_{f}"] = X[i, f]
            row["chosen"] = int(i == chosen_idx)
            rows.append(row)

    df = pd.DataFrame(rows)
    gt = {"theta": theta, "C": C_eff, "n_features": d,
          "slate_size": slate_size, "context_strength": context_strength}
    return df, gt

## 3. Generate the two datasets

We generate two datasets from the *same* features and *same* θ:
- `df_ctx` — the **effect** dataset, `context_strength=1.0`, so **C ≠ 0** (real context effects).
- `df_base` — the **baseline** dataset, `context_strength=0.0`, so **C = 0** (no context).

This prints the ground-truth θ and **C** — these are the numbers everything downstream is checked against. The dataframe is long-format: one row per item, keyed by `session_id`, with 40,000 rows = 8,000 slates × 5 items.

In [ ]:
df_ctx,  gt_ctx  = generate_cdm_data(context_strength=1.0, seed=0)  # C != 0
df_base, gt_base = generate_cdm_data(context_strength=0.0, seed=0)  # C  = 0

print("theta:", np.round(gt_ctx["theta"], 3))
print("C (ground-truth interaction matrix):")
print(np.round(gt_ctx["C"], 3))
print("effect shape:", df_ctx.shape, "| sessions:", df_ctx.session_id.nunique())
df_ctx.head(10)

theta: [-1.   1.   0.5]
C (ground-truth interaction matrix):
[[-1.424  1.264 -0.871]
 [-0.259 -0.075 -0.741]
 [-1.368  0.649  0.361]]
effect shape: (40000, 6) | sessions: 8000


,session_id,item_in_slate,feat_0,feat_1,feat_2,chosen
0,0,0,0.125730,-0.132105,0.640423,1
1,0,1,0.104900,-0.535669,0.361595,0
2,0,2,1.304000,0.947081,-0.703735,0
3,0,3,-1.265421,-0.623274,0.041326,0
4,0,4,-2.325031,-0.218792,-1.245911,0
5,1,0,-0.544259,-0.316300,0.411631,0
6,1,1,1.042513,-0.128535,1.366463,1
7,1,2,-0.665195,0.351510,0.903470,0
8,1,3,0.094012,-0.743499,-0.921725,0
9,1,4,-0.457726,0.220195,-1.009618,0


## 4. Sanity checks on the generated data

Two quick checks that the data is well-formed:
- **Exactly one winner per slate** — `chosen` should sum to 1 in every session.
- **Win-by-slot distribution** — since features are drawn i.i.d. for every slot, no slot is privileged, so both effect and baseline should be roughly flat (~1600 each). This flatness is *good*: it confirms there's no accidental positional artifact. The context effect shows up in *which item wins given its neighbours*, not in any fixed slot winning more often.

In [ ]:
per = df_ctx.groupby("session_id")["chosen"].sum()
print("chosen per slate: min={}, max={} (both should be 1)".format(per.min(), per.max()))

win_ctx  = df_ctx[df_ctx.chosen == 1].item_in_slate.value_counts().sort_index()
win_base = df_base[df_base.chosen == 1].item_in_slate.value_counts().sort_index()
print("win by slot (effect):  ", win_ctx.values)
print("win by slot (baseline):", win_base.values)

chosen per slate: min=1, max=1 (both should be 1)
win by slot (effect):   [1595 1624 1644 1573 1564]
win by slot (baseline): [1628 1650 1579 1563 1580]


## 5. Save the datasets to disk

Persist everything so later steps can read from disk rather than regenerating:
- `cdm_data_context.csv` — the effect dataset (C ≠ 0)
- `cdm_data_baseline.csv` — the baseline control (C = 0)
- `cdm_ground_truth.npz` — the exact θ and **C** used, i.e. the recovery target.

In [ ]:
df_ctx.to_csv("cdm_data_context.csv", index=False)
df_base.to_csv("cdm_data_baseline.csv", index=False)
np.savez("cdm_ground_truth.npz",
         theta=gt_ctx["theta"], C=gt_ctx["C"],
         n_features=gt_ctx["n_features"], slate_size=gt_ctx["slate_size"])
print("saved: cdm_data_context.csv, cdm_data_baseline.csv, cdm_ground_truth.npz")

saved: cdm_data_context.csv, cdm_data_baseline.csv, cdm_ground_truth.npz


## 6. Set up PyTorch

Import torch and pick the device (GPU if available, else CPU). Everything from here on — the models and training — runs in PyTorch.

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cpu


## 7. Collate: long dataframe → padded slate tensors

The model consumes whole slates, not individual rows. This groups the long dataframe by `session_id` and stacks each slate into tensors:
- `X` — item features, shape `(n_slates, max_slate, n_features)`
- `y` — a 1 marking the chosen item in each slate
- `mask` — marks which entries are real items (vs padding)

Slates are fixed at size 5 here, so nothing actually gets padded yet — but we build the masked/padded version anyway so the identical code still works once we move to variable-length slates later.

In [ ]:
def collate_slates(df, n_features=3):
    """
    Long-format df (one row per item) -> slate-batched padded tensors.

    Returns:
        X    : (n_slates, max_slate, n_features)  float  -- item features
        y    : (n_slates, max_slate)              float  -- 1 for chosen item
        mask : (n_slates, max_slate)              bool   -- True = real item
    """
    feat_cols = [f"feat_{f}" for f in range(n_features)]
    groups = list(df.groupby("session_id"))
    n_slates = len(groups)
    max_slate = df.groupby("session_id").size().max()

    X    = torch.zeros(n_slates, max_slate, n_features)
    y    = torch.zeros(n_slates, max_slate)
    mask = torch.zeros(n_slates, max_slate, dtype=torch.bool)

    for s, (_, g) in enumerate(groups):
        g = g.sort_values("item_in_slate")
        k = len(g)
        X[s, :k]    = torch.tensor(g[feat_cols].values, dtype=torch.float32)
        y[s, :k]    = torch.tensor(g["chosen"].values,  dtype=torch.float32)
        mask[s, :k] = True

    return X, y, mask

## 8. Run collate on both datasets

Apply the collate to the effect and baseline data. The shapes confirm `(8000, 5, 3)` for features, one chosen item per slate, and the mask all-True (since every slate is full at size 5).

In [ ]:
X_ctx,  y_ctx,  mask_ctx  = collate_slates(df_ctx)
X_base, y_base, mask_base = collate_slates(df_base)

print("X_ctx :", X_ctx.shape)     # (8000, 5, 3)
print("y_ctx :", y_ctx.shape)     # (8000, 5)
print("mask  :", mask_ctx.shape)  # (8000, 5)
print("chosen per slate (should all be 1):",
      y_ctx.sum(dim=1).min().item(), y_ctx.sum(dim=1).max().item())
print("all items real (fixed slates, mask all True):", mask_ctx.all().item())

X_ctx : torch.Size([8000, 5, 3])
y_ctx : torch.Size([8000, 5])
mask  : torch.Size([8000, 5])
chosen per slate (should all be 1): 1.0 1.0
all items real (fixed slates, mask all True): True


## 9. Train/test split

An 80/20 split into train and test, applied identically to both the effect and baseline datasets so the C=0 control stays a clean parallel. This gives 6,400 train / 1,600 test slates each.

In [ ]:
def train_test_split_slates(X, y, mask, frac=0.8, seed=0):
    n = X.shape[0]
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(n, generator=g)
    n_train = int(frac * n)
    tr, te = perm[:n_train], perm[n_train:]
    return (X[tr], y[tr], mask[tr]), (X[te], y[te], mask[te])

(Xtr_c, ytr_c, mtr_c), (Xte_c, yte_c, mte_c) = train_test_split_slates(X_ctx,  y_ctx,  mask_ctx)
(Xtr_b, ytr_b, mtr_b), (Xte_b, yte_b, mte_b) = train_test_split_slates(X_base, y_base, mask_base)

print("effect   train:", Xtr_c.shape[0], "test:", Xte_c.shape[0])
print("baseline train:", Xtr_b.shape[0], "test:", Xte_b.shape[0])

effect   train: 6400 test: 1600
baseline train: 6400 test: 1600


## 10. The self-attention choice model

This is the flexible model whose recovery we want to test. Each item is embedded, then passed through a `TransformerEncoder` so **every item can attend to every other item in the slate** — giving it the architectural capacity to represent cross-item context effects. A final linear layer scores each item; padded slots are masked to −∞ before the softmax.

The key point: with ~17,000 parameters this model has *far* more capacity than the 12-number true model. So if it under-recovers the effect, that's an *identification* failure, not a capacity limit.

In [ ]:
import torch.nn as nn

class AttentionChoiceModel(nn.Module):
    def __init__(self, n_features=3, d_model=32, n_heads=4, n_layers=2):
        super().__init__()
        self.embed = nn.Linear(n_features, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=64, batch_first=True, dropout=0.0)
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.score = nn.Linear(d_model, 1)

    def forward(self, X, mask):
        # X: (B, S, F)  mask: (B, S) True=real
        h = self.embed(X)                          # (B, S, d)
        pad = ~mask                                # True = padding to ignore
        h = self.encoder(h, src_key_padding_mask=pad)
        logits = self.score(h).squeeze(-1)         # (B, S)
        logits = logits.masked_fill(pad, float("-inf"))
        return logits                              # feed to softmax over slate

model = AttentionChoiceModel().to(device)
n_params = sum(p.numel() for p in model.parameters())
print("attention model params:", n_params)

attention model params: 17249


## 11. The training loop

Standard listwise choice training: for each slate, softmax over the item logits and apply cross-entropy against the index of the truly chosen item. Prints train loss, test loss, and test accuracy every few epochs. Random-guess accuracy on 5-item slates is 0.20, so anything well above that means the model is learning real choice structure.

In [ ]:
import torch.nn.functional as F

def train_model(model, Xtr, ytr, mtr, Xte, yte, mte,
                epochs=30, lr=1e-3, batch=256, seed=0):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    Xtr, ytr, mtr = Xtr.to(device), ytr.to(device), mtr.to(device)
    Xte, yte, mte = Xte.to(device), yte.to(device), mte.to(device)

    target_tr = ytr.argmax(dim=1)   # index of chosen item per slate
    target_te = yte.argmax(dim=1)
    n = Xtr.shape[0]
    g = torch.Generator().manual_seed(seed)

    for ep in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(n, generator=g)
        tot = 0.0
        for i in range(0, n, batch):
            idx = perm[i:i+batch]
            logits = model(Xtr[idx], mtr[idx])
            loss = F.cross_entropy(logits, target_tr[idx])
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(idx)
        if ep % 5 == 0 or ep == 1:
            model.eval()
            with torch.no_grad():
                te_logits = model(Xte, mte)
                te_loss = F.cross_entropy(te_logits, target_te).item()
                acc = (te_logits.argmax(1) == target_te).float().mean().item()
            print(f"epoch {ep:2d} | train loss {tot/n:.4f} | "
                  f"test loss {te_loss:.4f} | test acc {acc:.3f}")
    return model

## 12. Train the attention model on the effect data

Train the attention model on the **C ≠ 0** data. Watch test accuracy climb to ~0.81 — the model predicts the chosen item 4 out of 5 times. (The nested-tensor warning is harmless: with fixed-size slates nothing is actually padded.)

In [ ]:
torch.manual_seed(0)
model_ctx = AttentionChoiceModel().to(device)
model_ctx = train_model(model_ctx, Xtr_c, ytr_c, mtr_c, Xte_c, yte_c, mte_c)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


epoch  1 | train loss 1.4571 | test loss 1.2127 | test acc 0.539
epoch  5 | train loss 0.5297 | test loss 0.5317 | test acc 0.805
epoch 10 | train loss 0.4642 | test loss 0.5044 | test acc 0.798
epoch 15 | train loss 0.4431 | test loss 0.5063 | test acc 0.794
epoch 20 | train loss 0.4325 | test loss 0.4925 | test acc 0.808
epoch 25 | train loss 0.4277 | test loss 0.4912 | test acc 0.810
epoch 30 | train loss 0.4149 | test loss 0.4992 | test acc 0.806


## 13. Train the same model on the baseline (C = 0) data

Train the identical architecture on the **C = 0** data, where the only signal is each item's own features. Accuracy plateaus much lower, around 0.57, almost immediately — there's simply no cross-item structure to learn. 

The gap between this (~0.57) and the effect run (~0.81) is entirely attributable to the context effect. That's the first confirmation the effect is real and substantial.

In [ ]:
torch.manual_seed(0)
model_base = AttentionChoiceModel().to(device)
model_base = train_model(model_base, Xtr_b, ytr_b, mtr_b, Xte_b, yte_b, mte_b)

epoch  1 | train loss 1.2212 | test loss 1.1253 | test acc 0.555
epoch  5 | train loss 1.0913 | test loss 1.0941 | test acc 0.564
epoch 10 | train loss 1.0871 | test loss 1.0929 | test acc 0.569
epoch 15 | train loss 1.0814 | test loss 1.0927 | test acc 0.576
epoch 20 | train loss 1.0782 | test loss 1.0965 | test acc 0.568
epoch 25 | train loss 1.0719 | test loss 1.1031 | test acc 0.561
epoch 30 | train loss 1.0681 | test loss 1.1114 | test acc 0.563


## 14. The CDM yardstick model

This is the **correctly-specified** model: it learns θ and **C** directly and computes utilities with the *exact same formula* the generator used (`x_j^T C x_i` summed over neighbours, via einsum). Only 12 parameters.

Because its form matches the truth, it *should* recover **C** almost perfectly — this is our ceiling. It's the benchmark that tells us the effect is fully recoverable in principle, so any shortfall by the attention model is the attention model's fault, not the data's.

In [ ]:
class CDMChoiceModel(nn.Module):
    """
    Correctly-specified CDM. Learns theta (F,) and C (F,F) directly.
    Utility for item i in slate:  theta . x_i + sum_{j!=i} x_j^T C x_i
    This mirrors the generator exactly, so it's the recovery ceiling.
    """
    def __init__(self, n_features=3):
        super().__init__()
        self.theta = nn.Parameter(torch.zeros(n_features))
        self.C     = nn.Parameter(torch.zeros(n_features, n_features))

    def forward(self, X, mask):
        # X: (B, S, F)  mask: (B, S) True=real
        base = X @ self.theta                      # (B, S)
        # M[b,j,i] = x_j^T C x_i
        XC = X @ self.C                            # (B, S, F)
        M  = torch.einsum("bjf,bif->bji", XC, X)   # (B, S, S)
        # sum over j != i, respecting mask (ignore padded j)
        realj = mask.unsqueeze(-1).float()         # (B, S, 1) over j
        Msum  = (M * realj).sum(dim=1)             # sum over j -> (B, S)
        diag  = torch.diagonal(M, dim1=1, dim2=2)  # M[b,i,i] -> (B, S)
        context = Msum - diag                      # remove j == i
        logits = base + context                    # (B, S)
        pad = ~mask
        logits = logits.masked_fill(pad, float("-inf"))
        return logits

## 15. Fit the yardstick on the effect data

Train the 12-parameter CDM. It needs more epochs and a higher learning rate but converges fast. Note it matches the 17k-parameter attention model on accuracy (~0.81) with a *slightly better* test loss — the correct model does as well with 1/1400th the parameters.

In [ ]:
torch.manual_seed(0)
model_cdm = CDMChoiceModel().to(device)
print("CDM yardstick params:", sum(p.numel() for p in model_cdm.parameters()))
model_cdm = train_model(model_cdm, Xtr_c, ytr_c, mtr_c, Xte_c, yte_c, mte_c,
                        epochs=100, lr=5e-2)

CDM yardstick params: 12
epoch  1 | train loss 0.7160 | test loss 0.4895 | test acc 0.813
epoch  5 | train loss 0.4438 | test loss 0.4545 | test acc 0.815
epoch 10 | train loss 0.4477 | test loss 0.4458 | test acc 0.819
epoch 15 | train loss 0.4468 | test loss 0.4591 | test acc 0.812
epoch 20 | train loss 0.4485 | test loss 0.4614 | test acc 0.809
epoch 25 | train loss 0.4462 | test loss 0.4628 | test acc 0.811
epoch 30 | train loss 0.4448 | test loss 0.4606 | test acc 0.811
epoch 35 | train loss 0.4470 | test loss 0.4617 | test acc 0.811
epoch 40 | train loss 0.4450 | test loss 0.4542 | test acc 0.810
epoch 45 | train loss 0.4468 | test loss 0.4526 | test acc 0.816
epoch 50 | train loss 0.4474 | test loss 0.4539 | test acc 0.809
epoch 55 | train loss 0.4454 | test loss 0.4476 | test acc 0.822
epoch 60 | train loss 0.4477 | test loss 0.4461 | test acc 0.817
epoch 65 | train loss 0.4467 | test loss 0.4563 | test acc 0.811
epoch 70 | train loss 0.4458 | test loss 0.4624 | test acc 0.811


## 16. Check: did the yardstick recover the true C?

Pull the learned θ and **C** out of the yardstick and compare to the ground truth, entry by entry. We expect θ ≈ [−1, 1, 0.5] and **Ĉ** tracking **C** closely, with a small relative Frobenius error and near-1.0 correlation across the 9 entries. This confirms the yardstick behaves as a proper ceiling — the effect *is* fully recoverable by the right model.

In [ ]:
C_true = gt_ctx["C"]                       # numpy (3,3)
C_hat  = model_cdm.C.detach().cpu().numpy()
theta_true = gt_ctx["theta"]
theta_hat  = model_cdm.theta.detach().cpu().numpy()

np.set_printoptions(precision=3, suppress=True)
print("theta true:", theta_true)
print("theta hat :", theta_hat)
print()
print("C true:\n", C_true)
print("\nC hat:\n", C_hat)

# how close, overall
fro_err = np.linalg.norm(C_hat - C_true) / np.linalg.norm(C_true)
corr = np.corrcoef(C_true.flatten(), C_hat.flatten())[0, 1]
print(f"\nrelative Frobenius error: {fro_err:.3f}")
print(f"correlation(true, hat) over the 9 entries: {corr:.3f}")

theta true: [-1.   1.   0.5]
theta hat : [-1.026  1.021  0.493]

C true:
 [[-1.424  1.264 -0.871]
 [-0.259 -0.075 -0.741]
 [-1.368  0.649  0.361]]

C hat:
 [[-1.398  1.25  -0.852]
 [-0.276 -0.056 -0.722]
 [-1.337  0.594  0.409]]

relative Frobenius error: 0.034
correlation(true, hat) over the 9 entries: 1.000


## 17. The controlled-swap probe

This is the core diagnostic. It measures how much of the true pairwise effect the attention model actually reproduces. For each of 3,000 trials:

1. Freeze a random **target** item *i*.
2. Build two slates identical except for **one swapped neighbour** (`x_jA` → `x_jB`); all fillers stay the same.
3. Compute **Δ_true** = the exact utility shift the true rule predicts for *i*, straight from **C**: $(x_j^B - x_j^A)^\top C x_i$.
4. Measure **Δ_attn** = how much the attention model's score for *i* actually moves between the two slates.

Because only the one neighbour changed and the target is identical in both, any shift *must* come from that neighbour — that's what isolates the pure context effect.

In [ ]:
def context_probe(attn_model, C_true, theta_true, n_features=3, slate_size=5,
                  n_trials=3000, seed=1):
    """
    For each trial:
      - draw a random target x_i and random filler items
      - build slate A (neighbour x_j^A) and slate B (neighbour x_j^B)
        identical except that ONE neighbour is swapped
      - Delta_true = (x_j^B - x_j^A)^T C x_i   (exact, from true C)
      - Delta_attn = attn_logit(i | B) - attn_logit(i | A)   (measured)
    Returns arrays of (Delta_true, Delta_attn) across trials.
    """
    rng = np.random.default_rng(seed)
    attn_model.eval()
    C_true = np.asarray(C_true)

    d_true_all, d_attn_all = [], []

    with torch.no_grad():
        for _ in range(n_trials):
            # target item i at slot 0, fillers at slots 2..end fixed across A/B
            x_i = rng.normal(0, 1, size=n_features)
            fillers = rng.normal(0, 1, size=(slate_size - 2, n_features))
            x_jA = rng.normal(0, 1, size=n_features)
            x_jB = rng.normal(0, 1, size=n_features)

            # true shift: only neighbour j changed -> (x_jB - x_jA)^T C x_i
            d_true = (x_jB - x_jA) @ (C_true @ x_i)

            # build the two slates: slot0=target, slot1=neighbour, rest=fillers
            slateA = np.vstack([x_i, x_jA, fillers])   # (S, F)
            slateB = np.vstack([x_i, x_jB, fillers])
            XA = torch.tensor(slateA, dtype=torch.float32, device=device).unsqueeze(0)
            XB = torch.tensor(slateB, dtype=torch.float32, device=device).unsqueeze(0)
            m  = torch.ones(1, slate_size, dtype=torch.bool, device=device)

            logitA = attn_model(XA, m)[0, 0].item()   # target is slot 0
            logitB = attn_model(XB, m)[0, 0].item()
            d_attn = logitB - logitA

            d_true_all.append(d_true)
            d_attn_all.append(d_attn)

    return np.array(d_true_all), np.array(d_attn_all)

## 18. Run the probe and read the recovery

Fit a line through all 3,000 (Δ_true, Δ_attn) points. The **slope** is the headline recovery number: 1.0 = perfect recovery, 0 = the model ignores neighbours entirely. The **correlation** says whether it tracks the right *shape* of the effect.

Result: slope ≈ 0.43 — the attention model reproduces only ~43% of the true pairwise push, despite its strong accuracy. Note also the range compression: true shifts span ~±25 but the model's span only ~±11, so it systematically shrinks the effect toward zero.

In [ ]:
d_true, d_attn = context_probe(model_ctx, gt_ctx["C"], gt_ctx["theta"])

# slope of d_attn on d_true = fraction of the true push the attn model reproduces
slope = np.polyfit(d_true, d_attn, 1)[0]
corr  = np.corrcoef(d_true, d_attn)[0, 1]
mean_recovery = (d_attn / d_true)[np.abs(d_true) > 0.5].mean()  # avoid tiny denom

print(f"n trials: {len(d_true)}")
print(f"slope (d_attn ~ d_true)      : {slope:.3f}")
print(f"correlation(d_true, d_attn)  : {corr:.3f}")
print(f"mean per-trial recovery ratio: {mean_recovery:.3f}")
print(f"\nrange d_true: [{d_true.min():.2f}, {d_true.max():.2f}]")
print(f"range d_attn: [{d_attn.min():.2f}, {d_attn.max():.2f}]")

n trials: 3000
slope (d_attn ~ d_true)      : 0.425
correlation(d_true, d_attn)  : 0.681
mean per-trial recovery ratio: 0.504

range d_true: [-26.99, 24.77]
range d_attn: [-10.47, 11.00]


## 19. Validate the probe on the baseline (C = 0) model

Crucial control: run the *same probe* on the model trained with no context effect. Δ_true is still computed against the real **C**, but this model never saw any context during training — so its measured shifts should be unrelated to Δ_true, giving slope ≈ 0.

Result: slope ≈ 0.003, correlation ≈ 0.017 — essentially flat. This proves the probe only lights up when there's a genuine effect present, so the 0.43 on the effect model is a real signal, not something the probe manufactures.

In [ ]:
# Same probe, but on the model trained on C=0 data.
# d_true is still computed against the REAL C (the effect that exists in the
# effect-world), but model_base never saw any context effect during training.
# So its measured shifts should be ~unrelated to d_true -> slope near 0.
d_true_b, d_attn_b = context_probe(model_base, gt_ctx["C"], gt_ctx["theta"])

slope_b = np.polyfit(d_true_b, d_attn_b, 1)[0]
corr_b  = np.corrcoef(d_true_b, d_attn_b)[0, 1]

print("BASELINE (C=0) model probed against true C:")
print(f"  slope      : {slope_b:.3f}   (effect model was 0.425)")
print(f"  correlation: {corr_b:.3f}   (effect model was 0.681)")
print(f"  range d_attn: [{d_attn_b.min():.2f}, {d_attn_b.max():.2f}]")
print("\nExpectation: slope and corr both near 0 -> probe reads ~nothing")
print("when there is genuinely no context effect to recover.")

BASELINE (C=0) model probed against true C:
  slope      : 0.003   (effect model was 0.425)
  correlation: 0.017   (effect model was 0.681)
  range d_attn: [-2.62, 2.99]

Expectation: slope and corr both near 0 -> probe reads ~nothing
when there is genuinely no context effect to recover.


## 20. Multi-seed run for error bars

One training seed could be a lucky (or unlucky) draw. Here we retrain the attention model from scratch under 5 different seeds and re-run the probe each time, reporting mean ± std of the slope, correlation, and accuracy.

Result: slope **0.463 ± 0.018** across seeds — tight, and robustly below the LCL run's ~0.74. This turns the single number into a defensible finding: the attention model consistently recovers under half of the genuinely-pairwise CDM effect, even though pairwise interaction is exactly what attention is nominally built to capture.

In [ ]:
def multiseed_recovery(seeds=(0,1,2,3,4)):
    slopes, corrs, accs = [], [], []
    for sd in seeds:
        torch.manual_seed(sd)
        m = AttentionChoiceModel().to(device)
        m = train_model(m, Xtr_c, ytr_c, mtr_c, Xte_c, yte_c, mte_c,
                        epochs=30, seed=sd)  # same data, different init/shuffle
        dt, da = context_probe(m, gt_ctx["C"], gt_ctx["theta"], seed=100+sd)
        slopes.append(np.polyfit(dt, da, 1)[0])
        corrs.append(np.corrcoef(dt, da)[0,1])
        with torch.no_grad():
            acc = (m(Xte_c.to(device), mte_c.to(device)).argmax(1)
                   == yte_c.to(device).argmax(1)).float().mean().item()
        accs.append(acc)
        print(f"seed {sd}: slope={slopes[-1]:.3f}  corr={corrs[-1]:.3f}  acc={acc:.3f}")
    slopes, corrs, accs = map(np.array, (slopes, corrs, accs))
    print(f"\nslope      : {slopes.mean():.3f} ± {slopes.std():.3f}")
    print(f"correlation: {corrs.mean():.3f} ± {corrs.std():.3f}")
    print(f"accuracy   : {accs.mean():.3f} ± {accs.std():.3f}")
    return slopes, corrs, accs

slopes, corrs, accs = multiseed_recovery()

epoch  1 | train loss 1.4571 | test loss 1.2127 | test acc 0.539
epoch  5 | train loss 0.5297 | test loss 0.5317 | test acc 0.805
epoch 10 | train loss 0.4642 | test loss 0.5044 | test acc 0.798
epoch 15 | train loss 0.4431 | test loss 0.5063 | test acc 0.794
epoch 20 | train loss 0.4325 | test loss 0.4925 | test acc 0.808
epoch 25 | train loss 0.4277 | test loss 0.4912 | test acc 0.810
epoch 30 | train loss 0.4149 | test loss 0.4992 | test acc 0.806
seed 0: slope=0.438  corr=0.685  acc=0.806
epoch  1 | train loss 1.3479 | test loss 0.9984 | test acc 0.698
epoch  5 | train loss 0.5245 | test loss 0.5289 | test acc 0.794
epoch 10 | train loss 0.4621 | test loss 0.5036 | test acc 0.800
epoch 15 | train loss 0.4492 | test loss 0.4874 | test acc 0.806
epoch 20 | train loss 0.4337 | test loss 0.4899 | test acc 0.806
epoch 25 | train loss 0.4300 | test loss 0.5088 | test acc 0.808
epoch 30 | train loss 0.4244 | test loss 0.5069 | test acc 0.801
seed 1: slope=0.482  corr=0.731  acc=0.801
epoc